# Stronghold Survivors - Colab FX Batch (Phase 3)\n\nRun cells top-to-bottom. This notebook processes source frame sequences into game-ready transparent FX PNGs.\n\nExpected input folders in Google Drive:\n- `tower_evolution/`\n- `elite_death/`\n- `boss_death/`\n- `cannon_impact/`\n- `energy_impact/`\n\nEach folder should contain a PNG frame sequence (`...001.png`, `...002.png`, etc.).

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')

In [ ]:
!pip -q install pillow

In [ ]:
# ----- CONFIG -----\nDRIVE_ROOT = '/content/drive/MyDrive/stronghold_fx'\nINPUT_ROOT = f'{DRIVE_ROOT}/src'\nOUTPUT_ROOT = f'{DRIVE_ROOT}/out'\n\n# Optional BiRefNet command template. Leave empty for luma-only alpha.\n# Use placeholders: {input} and {output}\nRUN_BIREFNET_CMD = ''\n\nJOBS = [\n    {'name': 'tower_evolution', 'downscale': 2, 'palette': 28, 'sheet_cols': 8},\n    {'name': 'elite_death', 'downscale': 2, 'palette': 28, 'sheet_cols': 8},\n    {'name': 'boss_death', 'downscale': 2, 'palette': 24, 'sheet_cols': 8},\n    {'name': 'cannon_impact', 'downscale': 2, 'palette': 24, 'sheet_cols': 8},\n    {'name': 'energy_impact', 'downscale': 2, 'palette': 24, 'sheet_cols': 8},\n]\n\nprint('INPUT_ROOT =', INPUT_ROOT)\nprint('OUTPUT_ROOT =', OUTPUT_ROOT)

In [ ]:
# ----- PIPELINE -----\nimport json\nimport math\nimport os\nimport re\nimport shlex\nimport subprocess\nfrom pathlib import Path\nfrom PIL import Image, ImageChops\n\nVALID_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tif', '.tiff'}\n\ndef natural_key(path: Path):\n    parts = re.split(r'(\\d+)', path.name.lower())\n    out = []\n    for p in parts:\n        out.append(int(p) if p.isdigit() else p)\n    return out\n\ndef list_frames(folder: Path):\n    if not folder.exists():\n        return []\n    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS], key=natural_key)\n\ndef resolve_alpha_frame(biref_dir: Path, source_frame: Path):\n    exact = biref_dir / source_frame.name\n    if exact.exists():\n        return exact\n    stem = source_frame.stem\n    for ext in ['.png', '.webp', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff']:\n        cand = biref_dir / f'{stem}{ext}'\n        if cand.exists():\n            return cand\n    return None\n\ndef alpha_from_image(img: Image.Image):\n    if img.mode in ('RGBA', 'LA'):\n        return img.getchannel('A').convert('L')\n    return img.convert('L')\n\ndef luma_alpha(src_rgb: Image.Image, threshold=22, gamma=1.15, gain=1.2):\n    gray = src_rgb.convert('L')\n    threshold = max(0, min(254, int(threshold)))\n    gamma = max(0.01, float(gamma))\n    gain = max(0.01, float(gain))\n    inv = float(255 - threshold)\n    def curve(px):\n        if px <= threshold:\n            return 0\n        normalized = (float(px) - float(threshold)) / inv\n        mapped = (normalized ** gamma) * 255.0 * gain\n        return int(max(0.0, min(255.0, mapped)))\n    return gray.point(curve, mode='L')\n\ndef palette_reduce(img: Image.Image, colors=24):\n    colors = max(2, min(256, int(colors)))\n    pal = img.convert('RGBA').quantize(colors=colors, method=Image.MEDIANCUT)\n    return pal.convert('RGBA')\n\ndef make_sheet(frames, output_path: Path, cols=8):\n    if not frames:\n        return\n    first = Image.open(frames[0]).convert('RGBA')\n    fw, fh = first.size\n    cols = max(1, int(cols))\n    rows = int(math.ceil(len(frames) / float(cols)))\n    sheet = Image.new('RGBA', (fw * cols, fh * rows), (0, 0, 0, 0))\n    for i, frame_path in enumerate(frames):\n        img = Image.open(frame_path).convert('RGBA')\n        x = (i % cols) * fw\n        y = (i // cols) * fh\n        sheet.paste(img, (x, y))\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    sheet.save(output_path)\n\ndef run_job(job):\n    name = job['name']\n    source_dir = Path(INPUT_ROOT) / name\n    output_dir = Path(OUTPUT_ROOT) / name\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    frames = list_frames(source_dir)\n    if not frames:\n        print(f'[SKIP] no frames: {source_dir}')\n        return None\n\n    biref_dir = None\n    if RUN_BIREFNET_CMD.strip():\n        biref_dir = output_dir / '_biref_alpha'\n        biref_dir.mkdir(parents=True, exist_ok=True)\n        cmd = RUN_BIREFNET_CMD.format(input=str(source_dir), output=str(biref_dir))\n        print(f'[BiRefNet] {name}')\n        subprocess.run(cmd, shell=True, check=True)\n\n    processed = []\n    missing = []\n    for source in frames:\n        src = Image.open(source).convert('RGBA')\n        luma = luma_alpha(src.convert('RGB'))\n        biref_alpha = Image.new('L', src.size, 0)\n        if biref_dir is not None:\n            af = resolve_alpha_frame(biref_dir, source)\n            if af is not None:\n                matte = Image.open(af)\n                biref_alpha = alpha_from_image(matte).resize(src.size, Image.BILINEAR)\n            else:\n                missing.append(source.name)\n\n        final_alpha = ImageChops.lighter(biref_alpha, luma)\n        out = src.copy()\n        out.putalpha(final_alpha)\n\n        downscale = int(job.get('downscale', 1))\n        if downscale > 1:\n            w, h = out.size\n            out = out.resize((max(1, w // downscale), max(1, h // downscale)), Image.NEAREST)\n\n        palette = int(job.get('palette', 0))\n        if palette > 0:\n            out = palette_reduce(out, palette)\n\n        out_path = output_dir / f'{source.stem}.png'\n        out.save(out_path)\n        processed.append(out_path)\n\n    if int(job.get('sheet_cols', 0)) > 0:\n        make_sheet(processed, output_dir / 'sheet.png', int(job['sheet_cols']))\n\n    report = {\n        'job': name,\n        'source_dir': str(source_dir),\n        'output_dir': str(output_dir),\n        'processed_count': len(processed),\n        'missing_biref_frames': missing,\n        'downscale': int(job.get('downscale', 1)),\n        'palette': int(job.get('palette', 0)),\n    }\n    (output_dir / 'batch_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')\n    print(f'[OK] {name}: {len(processed)} frames')\n    return report\n\nall_reports = []\nfor job in JOBS:\n    rep = run_job(job)\n    if rep:\n        all_reports.append(rep)\n\nPath(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)\nsummary_path = Path(OUTPUT_ROOT) / 'phase3_summary.json'\nsummary_path.write_text(json.dumps(all_reports, indent=2), encoding='utf-8')\nprint('Summary:', summary_path)

In [ ]:
# Package all outputs into one zip for easy handoff\nimport shutil\nzip_base = '/content/stronghold_phase3_fx_batch'\nzip_path = shutil.make_archive(zip_base, 'zip', OUTPUT_ROOT)\nprint('ZIP ready:', zip_path)

In [ ]:
# Optional: copy zip back to Drive\nimport shutil\ndrive_zip = f'{DRIVE_ROOT}/stronghold_phase3_fx_batch.zip'\nshutil.copy('/content/stronghold_phase3_fx_batch.zip', drive_zip)\nprint('Copied to:', drive_zip)